# VALIDATOR

XML REQUEST TO XML SCHEMA

In [1]:
# imports

import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display
from pathlib import Path


In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")


OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AI
DeepSeek API Key exists and begins sk-
Groq API Key exists and begins gsk_
Grok API Key exists and begins xai-
OpenRouter API Key exists and begins sk-


In [3]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

openai = OpenAI()

# For Gemini, DeepSeek and Groq, we can use the OpenAI python client
# Because Google and DeepSeek have endpoints compatible with OpenAI
# And OpenAI allows you to change the base_url

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
deepseek_url = "https://api.deepseek.com"
groq_url = "https://api.groq.com/openai/v1"
grok_url = "https://api.x.ai/v1"
openrouter_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
deepseek = OpenAI(api_key=deepseek_api_key, base_url=deepseek_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [4]:
def read_directory_contents(base_path: str) -> str:
    """
    Recursively reads all files under base_path and returns a single string
    containing relative file paths followed by their contents.

    :param base_path: Root directory to scan
    :return: Aggregated string of file paths and contents
    """
    result_parts = []

    for root, _, files in os.walk(base_path):
        for file_name in files:
            full_path = os.path.join(root, file_name)
            rel_path = os.path.relpath(full_path, base_path)

            try:
                with open(full_path, 'r', encoding='utf-8', errors='ignore') as f:
                    content = f.read()
            except Exception as e:
                # Skip unreadable files but record the issue
                content = f"[Error reading file: {e}]"

            result_parts.append(f"File directory: {rel_path}\n\n{content}\n")

    return "\n".join(result_parts)

In [5]:

def read_all_files_to_string(directory: str, recursive: bool = True) -> str:
    """
    Reads all files in a directory and concatenates their contents into one string.
    Assumes all files are UTF-8 encoded.
    """
    base_path = Path(directory)

    if not base_path.exists():
        raise ValueError(f"Directory does not exist: {directory}")

    all_texts = []

    # Choose iterator
    files = base_path.rglob("*") if recursive else base_path.glob("*")

    for file_path in files:
        if file_path.is_file():
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    content = f.read()
                    all_texts.append(f"\n--- FILE: {file_path} ---\n")
                    all_texts.append(content)
            except Exception as e:
                print(f"Skipping {file_path}: {e}")

    return "".join(all_texts)

In [6]:
gpt_model = "gpt-5-mini"
system_prompt = """You are being given an XSD which which has been separated in different files but they are related together and they are forming the shcema as whole. The user will send you requests and 
your job is to validate those requests strictly to the schema give and to point out preciesly every minor detail that is not correct to the schema. Do not validate the sequence of the elements, it is
not important. The element SapReturn is actually optional, not mandatory, don't validate it. The declaration of the namespaces is not important too, don't validate it. Look more carefuly at the 
names to match, the data types, the length, and the cardinality of the fields. Point only to the elements where there is a problem. If everything is valid and you don't find anything, say so:    """
system_prompt = system_prompt + read_all_files_to_string("C:/Users/ME36352/MyFolder/REPOS/ACE/finance-accounting-integration-technical-invoices-financial-documents-v2/apps/finance-accounting-integration-technical-invoices-financial-documents-v2/SAP")
print(system_prompt)
request = """
 <sapzwsfindoc:SapZwsFinDoc xmlns:sapaddresses="http://www.ibm.com/xmlns/prod/websphere/j2ca/sap/sapaddresses2073361285" xmlns:sapcardinformation="http://www.ibm.com/xmlns/prod/websphere/j2ca/sap/sapcardinformation932265214" xmlns:sapcredittransfer="http://www.ibm.com/xmlns/prod/websphere/j2ca/sap/sapcredittransfer2124727546" xmlns:sapdata="http://www.ibm.com/xmlns/prod/websphere/j2ca/sap/sapdata1333925805" xmlns:sapdoccustomelements="http://www.ibm.com/xmlns/prod/websphere/j2ca/sap/sapdoccustomelements755363819" xmlns:sapdocs="http://www.ibm.com/xmlns/prod/websphere/j2ca/sap/sapdocs438412961" xmlns:sapdocumentlines="http://www.ibm.com/xmlns/prod/websphere/j2ca/sap/sapdocumentlines901811010" xmlns:sapfindoc="http://www.ibm.com/xmlns/prod/websphere/j2ca/sap/sapfindoc1249096900" xmlns:sapheader="http://www.ibm.com/xmlns/prod/websphere/j2ca/sap/sapheader6103334" xmlns:sapheadercustomelements="http://www.ibm.com/xmlns/prod/websphere/j2ca/sap/sapheadercustomelements2061585280" xmlns:sapmeta="http://www.ibm.com/xmlns/prod/websphere/j2ca/sap/sapmeta528807181" xmlns:sappartner="http://www.ibm.com/xmlns/prod/websphere/j2ca/sap/sappartner639908812" xmlns:sappartnercustomelements="http://www.ibm.com/xmlns/prod/websphere/j2ca/sap/sappartnercustomelements383215995" xmlns:sappaymentinstructions="http://www.ibm.com/xmlns/prod/websphere/j2ca/sap/sappaymentinstructions316354212" xmlns:sapreferences="http://www.ibm.com/xmlns/prod/websphere/j2ca/sap/sapreferences725672799.xsd" xmlns:sapreturn="http://www.ibm.com/xmlns/prod/websphere/j2ca/sap/sapreturn1478768624" xmlns:sapserviceproviderid="http://www.ibm.com/xmlns/prod/websphere/j2ca/sap/sapserviceproviderid1102181588" xmlns:saptechnicalreponse="http://www.ibm.com/xmlns/prod/websphere/j2ca/sap/saptechnicalreponse568329194" xmlns:sapthirdaddresses="http://www.ibm.com/xmlns/prod/websphere/j2ca/sap/sapthirdaddresses1000666237" xmlns:sapthirdparties="http://www.ibm.com/xmlns/prod/websphere/j2ca/sap/sapthirdparties366063943" xmlns:sapthirdpartycustomelements="http://www.ibm.com/xmlns/prod/websphere/j2ca/sap/sapthirdpartycustomelements1538637409" xmlns:sapzwsfindoc="http://www.ibm.com/xmlns/prod/websphere/j2ca/sap/sapzwsfindoc" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.ibm.com/xmlns/prod/websphere/j2ca/sap/sapzwsfindoc SapZwsFinDoc.xsd">
    <SapFinDoc>
        <SapDocs>
            <SapHeader>
                <DOCUMENT_DATE>2026-06-02</DOCUMENT_DATE>
                <DOCUMENT_TYPE>380</DOCUMENT_TYPE>
                <REFERENCE>XCRM-31511/46175</REFERENCE>
                <SapHeaderCustomElements>
                    <CUSTOM_KEY>CAS_REF_NUMBER</CUSTOM_KEY>
                    <CUSTOM_VALUE>0t7FT0000000jl7YAA</CUSTOM_VALUE>
                </SapHeaderCustomElements>
                <SapHeaderCustomElements>
                    <CUSTOM_KEY>CURRENCY</CUSTOM_KEY>
                    <CUSTOM_VALUE>CHF</CUSTOM_VALUE>
                </SapHeaderCustomElements>
                <SapHeaderCustomElements>
                    <CUSTOM_KEY>WRBTR</CUSTOM_KEY>
                    <CUSTOM_VALUE>50.00</CUSTOM_VALUE>
                </SapHeaderCustomElements>
                <SapHeaderCustomElements>
                    <CUSTOM_KEY>TOTAMTVATINCLUDED</CUSTOM_KEY>
                    <CUSTOM_VALUE>50.00</CUSTOM_VALUE>
                </SapHeaderCustomElements>
                <SapHeaderCustomElements>
                    <CUSTOM_KEY>TOTAMTVATEXCLUDED</CUSTOM_KEY>
                    <CUSTOM_VALUE>50.00</CUSTOM_VALUE>
                </SapHeaderCustomElements>
                <SapHeaderCustomElements>
                    <CUSTOM_KEY>SUMAMTVATEXCLUDED</CUSTOM_KEY>
                    <CUSTOM_VALUE>50.00</CUSTOM_VALUE>
                </SapHeaderCustomElements>
                <SapHeaderCustomElements>
                    <CUSTOM_KEY>CASE_HANDLER</CUSTOM_KEY>
                    <CUSTOM_VALUE>CAS Test BO Agent</CUSTOM_VALUE>
                </SapHeaderCustomElements>
                <SapHeaderCustomElements>
                    <CUSTOM_KEY>COMPANY_CODE</CUSTOM_KEY>
                    <CUSTOM_VALUE>1000</CUSTOM_VALUE>
                </SapHeaderCustomElements>
                <POSTING_DATE>20260602</POSTING_DATE>
            </SapHeader>
            <SapPartner>
                <SapAddresses>
                    <ADDRESS_TYPE>STD</ADDRESS_TYPE>
                    <COUNTRY>CH</COUNTRY>
                </SapAddresses>
                <LEGAL_ENTITY_NAME1>REGA</LEGAL_ENTITY_NAME1>
                <LEGAL_ENTITY_NAME2>REGA</LEGAL_ENTITY_NAME2>
                <SapReferences>
                    <REFERENCE_TYPE>SAPREF</REFERENCE_TYPE>
                    <REFERENCE>1200000004</REFERENCE>
                </SapReferences>
            </SapPartner>
            <SapPaymentInstructions>
                <PAYMENT_TERMS>Z010</PAYMENT_TERMS>
                <PAYMENT_MEANS_TYPE>30</PAYMENT_MEANS_TYPE>
                <REFERENCE>Dossier 14532492 Nusmir Beciri</REFERENCE>
                <SapCreditTransfer>
                    <IBAN>CH510022522595291301C</IBAN>
                    <SapServiceProviderId>
                        <CUSTOM_KEY>BANK_COUNTRY</CUSTOM_KEY>
                        <CUSTOM_VALUE>CH</CUSTOM_VALUE>
                    </SapServiceProviderId>
                    <SapServiceProviderId>
                        <CUSTOM_KEY>BANK_KEY</CUSTOM_KEY>
                        <CUSTOM_VALUE></CUSTOM_VALUE>
                    </SapServiceProviderId>
                </SapCreditTransfer>
            </SapPaymentInstructions>
            <SapDocumentLines>
                <ITEM_IDENTIFIER>FT0000000cDNYAY</ITEM_IDENTIFIER>
                <UNIT>999</UNIT>
                <QUANTITY>1</QUANTITY>
                <NET_AMOUNT>50.00</NET_AMOUNT>
                <VAT_RATE>0</VAT_RATE>
                <VAT_CODE>-</VAT_CODE>
                <SapDocCustomElements>
                    <CUSTOM_KEY>ITEM_TEXT</CUSTOM_KEY>
                    <CUSTOM_VALUE>Dossier 14532492 Service</CUSTOM_VALUE>
                </SapDocCustomElements>
                <SapDocCustomElements>
                    <CUSTOM_KEY>PRODUCT_CODE</CUSTOM_KEY>
                    <CUSTOM_VALUE>103</CUSTOM_VALUE>
                </SapDocCustomElements>
                <SapDocCustomElements>
                    <CUSTOM_KEY>B2B_PRODUCT_DESC</CUSTOM_KEY>
                    <CUSTOM_VALUE>TCS ETI Travel</CUSTOM_VALUE>
                </SapDocCustomElements>
                <SapDocCustomElements>
                    <CUSTOM_KEY>PRODUCT_FAMILY</CUSTOM_KEY>
                    <CUSTOM_VALUE>B2C</CUSTOM_VALUE>
                </SapDocCustomElements>
                <SapDocCustomElements>
                    <CUSTOM_KEY>MEMBER</CUSTOM_KEY>
                    <CUSTOM_VALUE>1</CUSTOM_VALUE>
                </SapDocCustomElements>
                <SapDocCustomElements>
                    <CUSTOM_KEY>MOTOR_VARIANT</CUSTOM_KEY>
                    <CUSTOM_VALUE>MOT</CUSTOM_VALUE>
                </SapDocCustomElements>
                <SapDocCustomElements>
                    <CUSTOM_KEY>COVER_AREA</CUSTOM_KEY>
                    <CUSTOM_VALUE>EU</CUSTOM_VALUE>
                </SapDocCustomElements>
                <SapDocCustomElements>
                    <CUSTOM_KEY>PRODUCT_VERSION</CUSTOM_KEY>
                    <CUSTOM_VALUE>2019</CUSTOM_VALUE>
                </SapDocCustomElements>
                <SapDocCustomElements>
                    <CUSTOM_KEY>FAMILY_VARIANT</CUSTOM_KEY>
                    <CUSTOM_VALUE>FAM</CUSTOM_VALUE>
                </SapDocCustomElements>
                <SapDocCustomElements>
                    <CUSTOM_KEY>COVER_TYPE</CUSTOM_KEY>
                    <CUSTOM_VALUE>STD</CUSTOM_VALUE>
                </SapDocCustomElements>
                <SapDocCustomElements>
                    <CUSTOM_KEY>YOUNGSTER</CUSTOM_KEY>
                    <CUSTOM_VALUE>0</CUSTOM_VALUE>
                </SapDocCustomElements>
                <SapDocCustomElements>
                    <CUSTOM_KEY>FIRST_YEAR_FREE</CUSTOM_KEY>
                    <CUSTOM_VALUE>0</CUSTOM_VALUE>
                </SapDocCustomElements>
                <SapDocCustomElements>
                    <CUSTOM_KEY>SECTION</CUSTOM_KEY>
                    <CUSTOM_VALUE>ZH</CUSTOM_VALUE>
                </SapDocCustomElements>
                <SapDocCustomElements>
                    <CUSTOM_KEY>IDIT_COVER_CODE</CUSTOM_KEY>
                    <CUSTOM_VALUE>1000153</CUSTOM_VALUE>
                </SapDocCustomElements>
                <SapDocCustomElements>
                    <CUSTOM_KEY>SAP_COVER_CODE</CUSTOM_KEY>
                    <CUSTOM_VALUE>111000003</CUSTOM_VALUE>
                </SapDocCustomElements>
                <SapDocCustomElements>
                    <CUSTOM_KEY>B2B_COVER_CODE</CUSTOM_KEY>
                    <CUSTOM_VALUE>1000153</CUSTOM_VALUE>
                </SapDocCustomElements>
                <SapDocCustomElements>
                    <CUSTOM_KEY>DAMAGE_CODE</CUSTOM_KEY>
                    <CUSTOM_VALUE>1000136</CUSTOM_VALUE>
                </SapDocCustomElements>
                <SapDocCustomElements>
                    <CUSTOM_KEY>CAS_SERVICE_CODE</CUSTOM_KEY>
                    <CUSTOM_VALUE>710220</CUSTOM_VALUE>
                </SapDocCustomElements>
                <SapDocCustomElements>
                    <CUSTOM_KEY>B2B_COVER_DESC</CUSTOM_KEY>
                    <CUSTOM_VALUE>Additional Costs For Final Return</CUSTOM_VALUE>
                </SapDocCustomElements>
                <SapDocCustomElements>
                    <CUSTOM_KEY>INCIDENT_DATE</CUSTOM_KEY>
                    <CUSTOM_VALUE>20260602</CUSTOM_VALUE>
                </SapDocCustomElements>
                <SapDocCustomElements>
                    <CUSTOM_KEY>ASSISTANCE_CASE_NUMBER</CUSTOM_KEY>
                    <CUSTOM_VALUE>14532492</CUSTOM_VALUE>
                </SapDocCustomElements>
                <SapDocCustomElements>
                    <CUSTOM_KEY>COUNTRY</CUSTOM_KEY>
                    <CUSTOM_VALUE>ES</CUSTOM_VALUE>
                </SapDocCustomElements>
                <SapDocCustomElements>
                    <CUSTOM_KEY>INCIDENT_CAUSE</CUSTOM_KEY>
                    <CUSTOM_VALUE>11601</CUSTOM_VALUE>
                </SapDocCustomElements>
                <SapDocCustomElements>
                    <CUSTOM_KEY>PRIMARY_CAUSE_OF_LOSS</CUSTOM_KEY>
                    <CUSTOM_VALUE>11600</CUSTOM_VALUE>
                </SapDocCustomElements>
                <SapDocCustomElements>
                    <CUSTOM_KEY>PAYMENT_BREAKDOWN</CUSTOM_KEY>
                    <CUSTOM_VALUE>500092</CUSTOM_VALUE>
                </SapDocCustomElements>
                <SapDocCustomElements>
                    <CUSTOM_KEY>CNL_DEDUCT</CUSTOM_KEY>
                    <CUSTOM_VALUE></CUSTOM_VALUE>
                </SapDocCustomElements>
                <SapDocCustomElements>
                    <CUSTOM_KEY>PA_DEDUCT</CUSTOM_KEY>
                    <CUSTOM_VALUE>0.00</CUSTOM_VALUE>
                </SapDocCustomElements>
                <SapDocCustomElements>
                    <CUSTOM_KEY>INSURANCE_LIMIT</CUSTOM_KEY>
                    <CUSTOM_VALUE>3000.00</CUSTOM_VALUE>
                </SapDocCustomElements>
                <GROSS_AMOUNT>50.00</GROSS_AMOUNT>
            </SapDocumentLines>
        </SapDocs>
    </SapFinDoc>
    <OBJECT_SYSTEM>CAS</OBJECT_SYSTEM>
    <PROCESS_CONTROL>ASSISTANCE_CASE_PAYABLE_DOCUMENTS_ASYNC</PROCESS_CONTROL>
</sapzwsfindoc:SapZwsFinDoc>"""

You are being given an XSD which which has been separated in different files but they are related together and they are forming the shcema as whole. The user will send you requests and 
your job is to validate those requests strictly to the schema give and to point out preciesly every minor detail that is not correct to the schema. Do not validate the sequence of the elements, it is
not important. The element SapReturn is actually optional, not mandatory, don't validate it. The declaration of the namespaces is not important too, don't validate it. Look more carefuly at the 
names to match, the data types, the length, and the cardinality of the fields. Point only to the elements where there is a problem. If everything is valid and you don't find anything, say so:    
--- FILE: C:\Users\ME36352\MyFolder\REPOS\ACE\finance-accounting-integration-technical-invoices-financial-documents-v2\apps\finance-accounting-integration-technical-invoices-financial-documents-v2\SAP\SapAddresses2073361285

In [7]:
def call_claude(chat):
    messages = [
       {"role": "system", "content": system_prompt},
        {"role": "user", "content": chat}
    ]
    #messages = claude_system + request 
    #response = openai.responses.create(model=gpt_model, input=messages)
    #return response.output_text
    response = openai.chat.completions.create(model=gpt_model, messages=messages, reasoning_effort="high")
    return response.choices[0].message.content
    

In [8]:
print(call_claude(request))

I checked the instance against the provided schemas (not validating element order/sequence, namespace declarations, or SapReturn as you requested). I found one schema violation:

- Element: /SapZwsFinDoc/SapFinDoc/SapDocs/SapPaymentInstructions/REFERENCE
  - Value: "Dossier 14532492 Nusmir Beciri"
  - Actual length: 30 characters
  - Schema constraint: xsd:string with maxLength = 27
  - Problem: the element's content exceeds the allowed maxLength (30 > 27). Trim the value to 27 characters or less.

No other violations were found (data types, lengths, and cardinalities are OK for the other elements in the XML).
